# 🫀 실험 15 — 첫 **finding head**: MI 부위 다중라벨

**MedKOS / `notebooks/exp15_mi_localization_head.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 여기서부터 계측이 아니라 **제품 방향**이다

실험 1~14는 전부 5-superclass 분류기를 **자**로 쓴 것이었다. 자는 제품이 아니다.
이 실험은 **소견을 학습 타깃으로 올리는 첫 시도**이고, 배관을 바꾼다.

| | 실험 1~14 (계측) | **실험 15** |
|---|---|---|
| 라벨 | 5-superclass **단일**(`len(sc)==1`) | MI 부위 **다중라벨** |
| 데이터 | 16,244건 (다중 superclass 5,144건 버림) | **전량에서 MI 부위 라벨을 뽑는다** |
| 출력 | softmax | **sigmoid** |
| 손실 | sparse CE | **BCE** |
| 지표 | macro-F1, 동작점 정합 | **macro-AUPRC**, 소견별 민감도 |
| 임계값 | α 이분탐색 하나 | **소견별로 따로** |

## 🚦 비용과 중단 지점 — CELL 3에서 멈출 수 있게 짰다

1. **CELL 2: 새 캐시 빌드** (PTB-XL records100 재다운로드 + 전량 전처리, CPU **30~60분**).
   기존 `ptbxl_12lead_full.npz` 를 **덮지 않는다** — 실험10′·13·13b·14가 그 캐시에 의존하므로
   새 파일 `ptbxl_12lead_all.npz` 로 따로 쓴다.
2. **CELL 3: 인구조사(G0)** — 다중라벨로 바꾸면 부위별 표본이 얼마나 느는지 **실측**한다.
   **여기서 결과를 보고 GPU를 켤지 정한다.** 측벽이 여전히 안 되면 학습할 이유가 줄어든다.
3. **CELL 4: 학습** — 유도 구성 **2개**(`{I,II}`·`{12}`) × 5겹 = **10회**, GPU **~1시간**.

**유도 구성을 2개로 줄인 이유**: 잔여 결손 `D = AUC({12}) − AUC({I,II})` 계산에는 이 둘이면
충분하고, 첫 다중라벨 판이라 배관이 어디서 터질지 모른다. 잘 돌면 `{II}`·`{II,V1}`을 나중에 붙인다.

## 설계 결정 — **한 번에 하나만 바꾼다**

**백본은 실험10′·9b 것을 글자 그대로 쓴다.** 아키텍처를 같이 바꾸면 "다중라벨 전환 덕인지
구조 덕인지" 못 가른다(실험9b에서 `weighted`와 `film`을 갈라 검정한 것과 같은 원칙).
바뀌는 것은 **헤드와 손실뿐**이다.

**손실도 평범한 BCE로 시작한다.** 실험14가 "동작점 조정만으로는 부족하다"고 말했으니
`pos_weight`가 필요할 공산이 크지만, **가중 없는 판이 없으면 가중의 효과를 귀속할 수 없다.**
→ `pos_weight`는 **실험15b**로 큐잉한다(R2의 "하나씩" 원칙).

**언더샘플링은 쓰지 않는다.** 다중라벨에서는 "다수 클래스"가 유일하지 않아 레코드 단위
균형이 정의되지 않는다. 불균형 대응은 15b의 `pos_weight`가 맡는다.

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 성격 |
|---|---|---|
| **G0 인구조사** | 양성 n ≥ 50 인 부위만 학습 타깃. 미만은 **"탐지 대상 아님"으로 명시 보고** | 후벽은 탈락이 예상된다 — 그 문장 자체가 차별점 A의 산출물이다 |
| **붕괴 감시** | 어떤 부위든 AUPRC가 유병률 수준(무작위)으로 죽으면 **판정 무효** | 실험10-full의 교훈 |
| **P-1 ★** | 다중라벨 헤드의 부위별 AUPRC > **기준선 A**(실험10′ 5-superclass MI 확률의 같은 부위 AUPRC) | 소견을 타깃으로 올린 값어치가 있는가 |
| **P-2** | 하벽경색에서 민감도 0.90 고정 시 특이도 ≥ 실험14의 MI 전체(`{12}` 0.605) | R1을 소견 단위로 걸 수 있는가 |
| **P-3** | 잔여 결손 `D`의 부위 순서가 **실험13b와 같은 방향**(하벽 < 전중격) | 차별점 A가 학습 타깃이 바뀌어도 유지되는가 |

**P-3이 이 실험의 숨은 본체다.** 지금까지의 `D` 표는 "소견을 배운 적 없는 모델"에서 나왔다.
**배운 모델에서도 순서가 유지되면 차별점 A는 모델 선택과 무관한 성질**이 되고,
안 유지되면 표에 "이 모델 기준"이라는 단서를 영구히 붙여야 한다.

**P-1이 실패하는 것도 결과다.** "부위를 따로 배워도 5-superclass 전이보다 낫지 않다"면
소견 헤드의 값어치가 없다는 뜻이고, 라벨 트리 설계를 다시 봐야 한다.


In [ ]:
# CELL 1 — 설정
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험10′과 동일해야 하는 값 (백본·분할을 그대로 쓴다)
FS      = 100
CONFIGS = {"I+II": [0, 1], "12": list(range(12))}     # ★ 2개로 시작 — 넷은 나중에
K_FOLD  = 5
N_SEEDS = 1
EPOCHS  = 20
SEED0   = 20260801
BOOT    = 2000
NMIN    = 50            # G0 관문: 양성 50건 미만 부위는 타깃에서 제외

# MI 부위 후보 (PTB-XL diagnostic_subclass). 실제 채택은 CELL 3 인구조사가 정한다.
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
# 평면 사전배정 — 실험13b와 같은 근거(그 소견을 읽는 유도가 어느 평면인가)
SITE_PLANE = {"IMI": "전두면", "ILMI": "혼합", "IPMI": "혼합", "IPLMI": "혼합",
              "ASMI": "횡단면", "AMI": "횡단면", "ALMI": "혼합",
              "LMI": "혼합", "PMI": "횡단면"}

CONFIG = dict(exp="exp15_mi_localization_head", quest="ailab-2026-0015",
              parent_exp="exp10p_lead_cv",
              purpose="소견(MI 부위)을 학습 타깃으로 올리는 첫 finding head",
              change_one_thing="백본·분할·에폭은 실험10′과 동일. 헤드(softmax→sigmoid)와 손실(CE→BCE)만 교체",
              no_class_weighting="가중 없는 기준판. pos_weight는 실험15b로 분리(R2 하나씩 원칙)",
              configs=list(CONFIGS), sites=SITE_CANDIDATES, site_plane=SITE_PLANE,
              predictions={"G0": f"양성 n>={NMIN} 부위만 타깃, 미만은 탐지 대상 아님으로 보고",
                           "붕괴감시": "AUPRC가 유병률 수준으로 죽으면 판정 무효",
                           "P-1": "부위별 AUPRC > 기준선 A(실험10′ 5-superclass MI 확률)",
                           "P-2": "하벽경색 민감도 0.90에서 특이도 >= 0.605(실험14 MI {12})",
                           "P-3": "잔여 결손 D의 부위 순서가 실험13b와 같은 방향"},
              k_fold=K_FOLD, n_seeds=N_SEEDS, epochs=EPOCHS, fs=FS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp15_mi_loc_head", CONFIG, project=PROJECT)

# 기준선 A용으로 실험10′ 산출물을 찾아둔다
REG = os.path.join(PROJECT, "registry.jsonl")
PREV = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp10p_lead_cv" and os.path.isdir(r.get("dir", "")):
        PREV = r["dir"]
run.log(f"기준선 A(실험10′): {PREV if PREV else '⚠️ 못 찾음 — P-1 판정 불가'}")

### CELL 2 — 새 캐시 빌드 (전량)

**기존 캐시를 덮지 않는다.** `ptbxl_12lead_full.npz`(16,244건 · 단일 superclass)는
실험10′·13·13b·14가 재현에 쓰는 자산이다. 여기서는 **`ptbxl_12lead_all.npz`** 를 새로 만든다.

records100 재다운로드가 필요하다(CPU 30~60분). **이미 만들어져 있으면 건너뛴다.**

> 🔁 **중간에 죽어도 처음부터 다시 받지 않는다.** `wget -N` 이 이미 완전히 받은 파일은
> 건너뛰고 **크기가 어긋난(= 끊긴) 파일만** 다시 받는다. 오류가 나면 **이 셀을 그냥 다시
> 실행**하면 되고, 여러 번 반복해도 안전하다. 셀 안에서도 최대 6번까지 자동으로 이어받으며,
> 매 시도마다 `현재 n/21799` 를 찍는다. 다 못 받으면 캐시를 만들지 않고 멈춘다
> (반쯤 받은 데이터로 조용히 학습하는 것이 최악이다).
>
> 다운로드가 끝나면 전처리는 **로컬 읽기라 몇 분**이면 끝난다.
>
> ♻️ **이전 실행이 엉뚱한 위치에 받아뒀어도 다시 받지 않는다.** 예전 판의 `--cut-dirs` 가
> 틀려 `records100/` 계층 없이 흩어졌다면, 셀이 **파일을 옮겨서** 복구한다(같은 디스크라
> 몇 초). 받아둔 것은 하나도 버리지 않는다.


In [ ]:
# CELL 2 — 전량 캐시 (신규 파일 · 기존 캐시 보존 · 몇 번을 다시 돌려도 안전)
import wfdb, pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df  = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)
sub_of = scp[scp.diagnostic == 1].diagnostic_subclass.to_dict()

OLD_CACHE = run.data("ptbxl_12lead_full.npz")     # 건드리지 않는다(재현 자산)
CACHE     = run.data("ptbxl_12lead_all.npz")      # 이번에 만드는 것
if os.path.exists(OLD_CACHE):
    run.log(f"기존 캐시 보존 확인: {OLD_CACHE}")

# ── 다운로드는 **재개된다**. -N 이 이미 완전히 받은 파일을 건너뛰고, 크기가 다른
#    (= 끊긴) 파일만 다시 받는다. 중간에 죽어도 이 셀을 다시 실행하면 이어서 간다.
#    ★ --cut-dirs 는 반드시 3 — URL 경로가 /files/ptb-xl/1.0.3/records100/... 이므로
#      3을 잘라야 로컬이 records100/00000/... 이 되어 filename_lr 과 맞는다.
#      (4로 자르면 records100 계층이 사라져 전 레코드 읽기 실패한다.)
REC = os.path.join(PTB, "records100")

def n_dat():
    if not os.path.isdir(REC): return 0
    return sum(sum(1 for f in fs if f.endswith(".dat")) for _, _, fs in os.walk(REC))

# ── ♻️ 구조 복구: 예전 실행이 --cut-dirs 를 잘못 줘서 records100 계층 없이 받아뒀다면
#    **재다운로드하지 않고 옮겨온다**. 같은 파일시스템이라 몇 초면 끝난다.
import glob, shutil

def relocate_stray():
    os.makedirs(REC, exist_ok=True)
    srcs = []
    # (a) /content/ptbxl/00000, 01000 … 이 records100 밖에 흩어진 경우
    srcs += [d for d in sorted(glob.glob(os.path.join(PTB, "[0-9]" * 5)))
             if os.path.isdir(d)]
    # (b) /content/ptbxl/files/ptb-xl/1.0.3/records100/… 처럼 더 깊이 박힌 경우
    for d in sorted(glob.glob(os.path.join(PTB, "**", "records100"), recursive=True)):
        if os.path.abspath(d) != os.path.abspath(REC) and os.path.isdir(d):
            srcs += [x for x in sorted(glob.glob(os.path.join(d, "[0-9]" * 5)))
                     if os.path.isdir(x)]
    moved = 0
    for d in srcs:
        dst = os.path.join(REC, os.path.basename(d))
        if not os.path.isdir(dst):
            shutil.move(d, dst)
            moved += sum(1 for f in os.listdir(dst) if f.endswith(".dat"))
            continue
        for f in os.listdir(d):                      # 이미 있으면 더 완전한 쪽을 남긴다
            a, b = os.path.join(d, f), os.path.join(dst, f)
            if not os.path.exists(b):
                shutil.move(a, b); moved += int(f.endswith(".dat"))
            elif os.path.getsize(b) >= os.path.getsize(a):
                os.remove(a)                         # 목적지가 같거나 더 크다 → 중복
            else:
                shutil.move(a, b); moved += int(f.endswith(".dat"))   # 원본이 더 완전
        if not os.listdir(d): os.rmdir(d)
    return len(srcs), moved

nd, nf = relocate_stray()
if nd:
    run.log(f"♻️ 잘못된 위치의 폴더 {nd}개를 records100/ 으로 옮겼다 "
            f"(.dat {nf:,}개) — **재다운로드 없음**")

WANT = len(df)
if not os.path.exists(CACHE):
    for attempt in range(1, 7):
        have = n_dat()
        if have >= WANT:
            run.log(f"✅ records100 완비 {have:,}/{WANT:,}"); break
        run.log(f"[{attempt}/6] records100 내려받는 중 … 현재 {have:,}/{WANT:,}"
                + ("  (이어받기)" if have else ""))
        t0 = time.time()
        subprocess.run(["wget", "-q", "-r", "-N", "-np", "-nH", "--cut-dirs=3",
                        "--retry-connrefused", "--tries=5", "--timeout=30",
                        "-P", PTB, f"{BASE}/records100/"], check=False)
        run.log(f"    이번 시도 {time.time()-t0:.0f}s · 누적 {n_dat():,}건")
    have = n_dat()
    if have < WANT:
        raise RuntimeError(
            f"records100 이 {have:,}/{WANT:,} 만 받아졌습니다. **이 셀을 다시 실행하세요** — "
            "이미 받은 파일은 건너뛰고 모자란 것만 이어받습니다. 여러 번 반복해도 안전합니다.")

if not os.path.exists(CACHE):
    run.log("전처리 시작 (다운로드는 끝났고 여기부터는 로컬 읽기라 몇 분이면 끝난다)")
    t0 = time.time()
    n = len(df)
    X = np.zeros((n, 1000, 12), "float32")     # ★ 미리 잡는다 — list→stack 은 RAM 2배
    keep, folds, pids, eids, bad = [], [], [], [], []
    for i, (eid, row) in enumerate(df.iterrows()):
        try:
            sig, _ = wfdb.rdsamp(os.path.join(PTB, row.filename_lr))
        except Exception:
            bad.append(int(eid)); continue
        if sig.shape != (1000, 12):
            bad.append(int(eid)); continue
        X[len(keep)] = sig.astype("float32")
        keep.append(i); eids.append(int(eid))
        folds.append(int(row.strat_fold)); pids.append(int(row.patient_id))
        if (i + 1) % 4000 == 0:
            run.log(f"  {i+1:,}/{n:,} · {time.time()-t0:.0f}s")
    X = X[:len(keep)]
    if bad:
        run.log(f"⚠️ 읽기 실패 {len(bad)}건 (예: {bad[:5]}) — 캐시에서 제외한다")
    np.savez_compressed(CACHE, X=X, fold=np.array(folds), pid=np.array(pids),
                        eid=np.array(eids))
    run.log(f"저장 {CACHE} · X{X.shape} · {time.time()-t0:.0f}s")
    del X
else:
    run.log(f"전량 캐시 재사용: {CACHE}")

z = np.load(CACHE, allow_pickle=True)
X, FOLD10, PID, EID = z["X"], z["fold"], z["pid"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD                       # ★ 실험10′과 동일한 겹 규칙
dfa = df.loc[EID]
# ★ 두 층위를 모두 만든다.
#   codes  = scp_codes 의 **원본 코드**(ASMI·ALMI·ILMI·IPMI·IPLMI…) ← 부위 국소화는 여기 산다
#   subcls = diagnostic_subclass (MI는 IMI·AMI·LMI·PMI 넷뿐인 **거친** 묶음)
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
dfa["subcls"] = dfa.scp_codes.apply(
    lambda s: sorted({sub_of[k] for k in ast.literal_eval(s) if k in sub_of}))
run.log(f"X{X.shape} · 레코드 {len(EID):,} (기존 캐시 16,244 대비 "
        f"+{len(EID)-16244:,}건 회수)")
for k in range(K_FOLD):
    run.log(f"  겹 {k}: {(CV==k).sum():,}건")

### CELL 3 — 【G0】 다중라벨 인구조사 · **여기서 멈출 수 있다**

다중라벨로 바꾸면 부위별 표본이 얼마나 느는지 **실측**한다. 추측하지 않는다.
`n < 50` 부위는 학습 타깃에서 빼고 **"탐지 대상 아님"** 으로 보고한다.

> ⚠️ **층위 주의(첫 판에서 틀렸던 곳).** 부위 국소화는 **scp 원본 코드**(ASMI·ALMI·ILMI·
> IPMI·IPLMI…) 층위에 있다. `diagnostic_subclass` 는 MI를 **`{IMI, AMI, LMI, PMI}` 넷**으로
> 뭉개기 때문에 그걸로 세면 나머지가 전부 0이 된다. 여기서는 **코드 층위로 세고**,
> subclass 층위는 대조용으로 함께 찍는다.

**이 셀 출력을 보고 GPU를 켤지 정한다.** 측벽이 여전히 안 되면 학습의 값어치가 줄어든다.


In [ ]:
# CELL 3 — 인구조사 + 타깃 확정
# ★★ 부위 국소화는 **scp 원본 코드** 층위에 있다. diagnostic_subclass 는 MI를
#     {IMI, AMI, LMI, PMI} 넷으로 뭉개므로 ASMI·ALMI·ILMI·IPMI·IPLMI 가 사라진다.
#     (첫 판에서 subclass 로 읽어 전부 0이 나왔던 자리다 — 두 층위를 다 찍어 대조한다.)
run.log("【참고】 diagnostic_subclass 층위 — MI는 넷뿐이라 국소화가 뭉개진다")
for v in sorted({x for vs in dfa.subcls for x in vs} & set(SITE_CANDIDATES)):
    n = int(sum(v in vs for vs in dfa.subcls))
    run.log(f"    {v:<8}{n:>8,}")

run.log("\n【주분석】 scp 원본 코드 층위")
run.log(f"  {'부위':<8}{'평면':<7}{'양성 n':>8}{'유병률':>9}   채택")
SITES, census = [], {}
for site in SITE_CANDIDATES:
    m = np.array([site in c for c in dfa.codes])
    n = int(m.sum()); ok = n >= NMIN
    census[site] = {"n": n, "prev": float(m.mean()), "plane": SITE_PLANE[site], "적용": ok}
    if ok: SITES.append(site)
    run.log(f"  {site:<8}{SITE_PLANE[site]:<7}{n:>8,}{m.mean():>9.4f}   "
            + ("✅" if ok else f"⛔ 탐지 대상 아님 (n<{NMIN})"))

if not SITES:
    raise RuntimeError("타깃이 하나도 없습니다 — 코드 이름이 맞는지 확인하세요: "
                       + str(sorted({x for c in dfa.codes for x in c})[:40]))
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")
run.log(f"\n채택 부위 {len(SITES)}개: {SITES}")
run.log(f"라벨 행렬 {Ymul.shape} · 부위 하나 이상 양성인 레코드 "
        f"{int((Ymul.sum(1) > 0).sum()):,}건 · 전부 음성 {int((Ymul.sum(1) == 0).sum()):,}건")
run.log(f"레코드당 평균 양성 부위 {Ymul[Ymul.sum(1) > 0].sum(1).mean():.2f}개 "
        "(1보다 크면 다중라벨 전환이 실제로 값을 한다)")

# 평면별로 채점 가능한지 미리 확인 (P-3 이 성립하려면 양쪽에 하나씩은 있어야 한다)
fr = [s for s in SITES if SITE_PLANE[s] == "전두면"]
tr = [s for s in SITES if SITE_PLANE[s] == "횡단면"]
run.log(f"\nP-3 채점 가능 여부 — 전두면 {fr} · 횡단면 {tr}"
        + ("  ✅" if fr and tr else "  ⚠️ 한쪽이 비어 P-3 판정 불가"))

excluded = [s for s in SITE_CANDIDATES if s not in SITES]
if excluded:
    run.log(f"\n⛔ 범위 밖(차별점 A에 그대로 보고): "
            + ", ".join(f"{e}(n={census[e]['n']})" for e in excluded))
    run.log("   → 리샘플링으로 만들 수 없다. 데이터가 더 필요하거나 유도가 더 필요하다"
            "(후벽은 V7–V9 없는 표준 12유도의 구조적 한계).")
run.log("\n■ 여기서 출력 확인 후 CELL 4(GPU 학습)로 진행 여부를 정하세요.")

In [ ]:
# CELL 4 — 다중라벨 학습 (2구성 × 5겹 = 10회)
import tensorflow as tf
from tensorflow.keras import layers, models

NS = len(SITES)

def mask_of(cfg):
    m = np.zeros(12, "float32"); m[CONFIGS[cfg]] = 1.0
    return m

def build_head(seed):
    """★ 백본은 실험10′과 동일. 마지막 층만 sigmoid 다중라벨로 바꾼다."""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(NS, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")     # ★ 가중 없음 — 15b가 pos_weight를 맡는다
    return m

OOF = {c: np.zeros((len(EID), NS), "float32") for c in CONFIGS}
t0, done, total = time.time(), 0, len(CONFIGS) * K_FOLD * N_SEEDS
for c in CONFIGS:
    mk = mask_of(c)
    for k in range(K_FOLD):
        if run.load_arm(f"{c}_f{k}") is not None:
            OOF[c][np.where(CV == k)[0]] = run.load_arm(f"{c}_f{k}")
            done += N_SEEDS; run.log(f"  ⏭ {c} 겹{k} 체크포인트"); continue
        te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
        rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
        n_val = max(int(len(rest) * 0.12), 200)
        va, tr = rest[:n_val], rest[n_val:]
        acc = np.zeros((len(te), NS), "float32")
        for s in range(N_SEEDS):
            m = build_head(SEED0 + 100 * k + 15 + s)
            m.fit(X[tr] * mk, Ymul[tr], validation_data=(X[va] * mk, Ymul[va]),
                  epochs=EPOCHS, batch_size=128, verbose=0)
            acc += m.predict(X[te] * mk, batch_size=512, verbose=0)
            if k == 0 and s == 0:
                run.save_model(m, f"head_{c}")
            tf.keras.backend.clear_session(); done += 1
            if done == 1:
                per = time.time() - t0
                run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
        run.save_arm(f"{c}_f{k}", acc / N_SEEDS)
        OOF[c][te] = acc / N_SEEDS
        run.log(f"  {c} 겹{k} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
run.log(f"\n총 {time.time()-t0:.0f}s")

In [ ]:
# CELL 5 — 평가: 부위별 AUPRC · 잔여 결손 · 제약 하 동작점
from sklearn.metrics import average_precision_score, roc_auc_score

def spec_at_sens(score, pos, target=0.90):
    p, n = score[pos], score[~pos]
    if len(p) == 0 or len(n) == 0: return np.nan, np.nan, np.nan
    thr = float(np.quantile(p, 1.0 - target, method="lower"))
    return float((p >= thr).mean()), float((n < thr).mean()), float((score >= thr).mean())

# ── 기준선 A: 실험10′ 5-superclass 의 MI 확률 (같은 레코드에서만 비교)
BASE_A = {}
if PREV:
    old = np.load(run.data("ptbxl_12lead_full.npz"), allow_pickle=True)
    old_eid = old["eid"]; old_cv = (old["fold"] - 1) % K_FOLD
    pos_in_new = {int(e): i for i, e in enumerate(EID)}
    inter = np.array([i for i, e in enumerate(old_eid) if int(e) in pos_in_new])
    new_i = np.array([pos_in_new[int(old_eid[i])] for i in inter])
    run.log(f"기준선 A 비교 가능 레코드 {len(inter):,}건 (두 캐시의 교집합)")
    for c in CONFIGS:
        arr = np.zeros((len(old_eid), 5))
        for k in range(K_FOLD):
            a = np.load(os.path.join(PREV, "arms", f"fixed_{c}_f{k}", "probs.npy"))
            arr[np.where(old_cv == k)[0]] = a
        BASE_A[c] = (arr[inter][:, 3], new_i)          # MI 확률과 대응 인덱스
else:
    run.log("⚠️ 실험10′ 산출물이 없어 P-1 판정 불가")

R, collapse = {}, []
run.log("\n" + "=" * 112)
run.log("【부위별 성능】 AUPRC(주지표) · AUROC · 잔여 결손 D")
run.log("=" * 112)
run.log(f"  {'부위':<8}{'평면':<7}{'n':>7}{'유병률':>8}"
        + "".join(f"{'AUPRC ' + c:>14}" for c in CONFIGS)
        + f"{'D(AUROC)':>11}{'기준선A(12)':>13}")
for j, s in enumerate(SITES):
    y = Ymul[:, j].astype(bool)
    ap = {c: float(average_precision_score(y, OOF[c][:, j])) for c in CONFIGS}
    au = {c: float(roc_auc_score(y, OOF[c][:, j])) for c in CONFIGS}
    D = au["12"] - au["I+II"]
    base = np.nan
    if BASE_A:
        sc, ni = BASE_A["12"]
        base = float(average_precision_score(Ymul[ni, j].astype(bool), sc))
    prev_rate = float(y.mean())
    # 붕괴 감시: AUPRC가 유병률(무작위 수준)의 1.2배도 안 되면 죽은 것으로 본다
    for c in CONFIGS:
        if ap[c] < prev_rate * 1.2:
            collapse.append({"site": s, "config": c, "auprc": ap[c], "prev": prev_rate})
    R[s] = {"n": int(y.sum()), "prev": prev_rate, "auprc": ap, "auroc": au,
            "D": D, "baseline_A_auprc": base, "plane": SITE_PLANE[s]}
    run.log(f"  {s:<8}{SITE_PLANE[s]:<7}{int(y.sum()):>7,}{prev_rate:>8.4f}"
            + "".join(f"{ap[c]:>14.3f}" for c in CONFIGS)
            + f"{D:>+11.3f}{base:>13.3f}")

run.log("\n【붕괴 감시】 AUPRC가 유병률의 1.2배 미만 = 사실상 무작위")
if collapse:
    for x in collapse:
        run.log(f"  ❌ {x['site']} · {x['config']} · AUPRC {x['auprc']:.3f} "
                f"vs 유병률 {x['prev']:.3f}")
else:
    run.log("  ✅ 없음 — 판정이 유효하다")

run.log("\n【제약 하 동작점】 민감도 0.90 고정 (실험14와 같은 절차)")
run.log(f"  {'부위':<8}" + "".join(f"{'특이도 ' + c:>14}{'경보율 ' + c:>14}" for c in CONFIGS))
for j, s in enumerate(SITES):
    y = Ymul[:, j].astype(bool); cells = []
    for c in CONFIGS:
        se, sp, al = spec_at_sens(OOF[c][:, j], y)
        R[s].setdefault("op", {})[c] = {"sens": se, "spec": sp, "alarm": al}
        cells += [f"{sp:.3f}", f"{al:.3f}"]
    run.log(f"  {s:<8}" + "".join(f"{x:>14}" for x in cells))

In [ ]:
# CELL 6 — 사전등록 채점
MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}
rs = np.random.RandomState(SEED0)

def boot_diff(a_score, a_y, b_score, b_y, B=BOOT):
    """AUPRC 차의 부트스트랩(같은 재표본 축)."""
    out = np.empty(B); n = len(a_y)
    for t in range(B):
        i = rs.randint(0, n, n)
        if a_y[i].sum() < 5: out[t] = np.nan; continue
        out[t] = (average_precision_score(a_y[i], a_score[i])
                  - average_precision_score(b_y[i], b_score[i]))
    out = out[~np.isnan(out)]
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

run.log("\n" + "=" * 112)
run.log("【사전등록 채점】")
run.log("=" * 112)

# P-1: 부위별 AUPRC > 기준선 A
P1 = None
if BASE_A:
    sc, ni = BASE_A["12"]
    wins, tot = 0, 0
    run.log("  P-1 부위별 AUPRC (다중라벨 헤드 vs 실험10′ 5-superclass 전이)")
    for j, s in enumerate(SITES):
        yy = Ymul[ni, j].astype(bool)
        d, lo, hi = boot_diff(OOF["12"][ni, j], yy, sc, yy)
        sig = "★" if lo > 0 else ("✗" if hi < 0 else " ")
        wins += int(lo > 0); tot += 1
        run.log(f"    {s:<8} Δ={d:>+8.4f} [{lo:+.4f}, {hi:+.4f}] {sig}")
    P1 = True if wins > tot / 2 else (False if wins == 0 else None)
    run.log(f"  → {MARK[P1]} (유의하게 이긴 부위 {wins}/{tot})")
else:
    run.log("  P-1 → ⚠️ 판정불가")

# P-2: 하벽경색 민감도 0.90에서 특이도 >= 0.605
P2 = None
if "IMI" in R:
    sp = R["IMI"]["op"]["12"]["spec"]
    P2 = bool(sp >= 0.605)
    run.log(f"  P-2 IMI 민감도 0.90에서 특이도 {sp:.3f} ≥ 0.605(실험14 MI 전체) → {MARK[P2]}")
    run.log(f"      경보율 {R['IMI']['op']['12']['alarm']:.3f}")

# P-3: D 의 부위 순서가 실험13b와 같은 방향 (하벽 < 전중격)
P3 = None
front = [s for s in SITES if SITE_PLANE[s] == "전두면"]
trans = [s for s in SITES if SITE_PLANE[s] == "횡단면"]
if front and trans:
    mf = float(np.mean([R[s]["D"] for s in front]))
    mt = float(np.mean([R[s]["D"] for s in trans]))
    P3 = bool(mt > mf)
    run.log(f"  P-3 잔여 결손 방향 유지 → {MARK[P3]}  "
            f"전두면 {mf:+.4f}({front}) vs 횡단면 {mt:+.4f}({trans})")
    run.log(f"      실험13b: 전두면 −0.008 vs 횡단면 +0.053 — 부호가 같아야 한다")
else:
    run.log(f"  P-3 → ⚠️ 판정불가 (전두면 {front} · 횡단면 {trans})")

if collapse:
    verdict = ("판정 무효 — 붕괴한 부위가 있다("
               + ", ".join(f"{x['site']}·{x['config']}" for x in collapse)
               + "). 가중 없는 BCE가 희소 부위를 못 배웠다는 뜻이므로 "
                 "**실험15b(pos_weight)로 넘긴다**")
    P1 = P2 = P3 = None
elif P1 is True and P3 is True:
    verdict = ("확증 — 소견을 타깃으로 올린 값어치가 있고(P-1), 차별점 A의 평면 순서가 "
               "학습 타깃이 바뀌어도 유지된다(P-3). **표에서 '이 모델 기준' 단서를 뗄 수 있다**")
elif P1 is True:
    verdict = "부분 확증 — 소견 헤드는 낫지만 평면 순서는 재확인이 필요하다"
elif P1 is False:
    verdict = ("기각 — 부위를 따로 배워도 5-superclass 전이보다 낫지 않다. "
               "소견 헤드의 값어치를 다시 따져야 하고, 라벨 트리 설계를 재검토한다")
else:
    verdict = "미결 — 방향은 맞을 수 있으나 CI가 0을 걸친다"
run.log(f"\n▶ {verdict}")
run.log("=" * 112)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.6, 0.5 * len(SITES) + 1.6))
ys = np.arange(len(SITES)); w = 0.36
for i, c in enumerate(CONFIGS):
    ax.barh(ys + (i - 0.5) * w, [R[s]["auprc"][c] for s in SITES], w, label=c)
ax.plot([R[s]["prev"] for s in SITES], ys, "k.", ms=7, label="유병률(무작위 수준)")
ax.set_yticks(ys); ax.set_yticklabels([f"{s} (n={R[s]['n']:,})" for s in SITES], fontsize=8)
ax.set_xlabel("AUPRC"); ax.legend(fontsize=8)
ax.set_title("실험15 — MI 부위 다중라벨 헤드", fontsize=10)
plt.tight_layout(); run.save_fig("mi_site_auprc", fig); plt.show()

run.save_json("evaluation", {"census": census, "sites": SITES, "result": R,
                             "collapse": collapse, "P-1": P1, "P-2": P2, "P-3": P3,
                             "verdict": verdict})

macro = float(np.mean([R[s]["auprc"]["12"] for s in SITES]))
result = {"week": 2, "exp_id": "exp15_mi_loc_head", "quest": "ailab-2026-0015",
          "task": "MI 부위 다중라벨 헤드 — 소견을 학습 타깃으로 올리는 첫 시도",
          "split": "inter", "metric": "macro_auprc_sites_12lead",
          "value": round(macro, 4), "passed": bool(P1 is True and not collapse),
          "date": time.strftime("%Y-%m-%d"), "k_fold": K_FOLD, "n_seeds": N_SEEDS,
          "n_records": int(len(EID)), "census": census, "sites": SITES,
          "subgroups": R, "degraded_classes": [f"{x['site']}·{x['config']}" for x in collapse],
          "P-1": P1, "P-2": P2, "P-3": P3, "verdict": verdict,
          "summary": (f"부위 {len(SITES)}개 · macro-AUPRC({{12}}) {macro:.3f} · "
                      f"P-1 {MARK[P1]} P-2 {MARK[P2]} P-3 {MARK[P3]} · "
                      + verdict.split(' —')[0])}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp15_mi_localization_head.ipynb \\
      --quest ailab-2026-0015 --step "exp15-mi-localization-head" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

**붕괴 감시부터 본다.** AUPRC가 유병률의 1.2배도 안 되면 그 부위는 무작위와 같고,
macro 비교는 거짓이 된다 → **판정 무효**, 실험15b로 넘긴다.

| P-1 | P-3 | 뜻 | 다음 |
|---|---|---|---|
| ✅ | ✅ | 소견 헤드가 값어치를 하고 **차별점 A가 모델과 무관** | `{II}`·`{II,V1}` 확장 → 라벨 트리 전체(2단계) |
| ✅ | ❌ | 성능은 낫지만 평면 순서가 바뀐다 | 차별점 A 표에 **"이 모델 기준"** 단서를 영구히 붙인다 |
| ❌ | — | 부위를 따로 배워도 전이보다 못하다 | 라벨 트리 설계 재검토. 소견 헤드가 답이 아닐 수 있다 |
| 무효 | — | 희소 부위가 죽었다 | **15b(`pos_weight`)** — R2의 두 번째 단계 |

## 이 실험이 남기는 것

1. **다중라벨 배관** — sigmoid·BCE·소견별 임계값·macro-AUPRC. 2~3단계가 전부 이걸 쓴다.
2. **전량 캐시** `ptbxl_12lead_all.npz` — 앞으로 모든 소견 실험의 기반.
3. **범위 밖 소견 목록** — "이 모델은 무엇을 탐지 대상으로도 삼지 않는가"의 근거.

## 한계 — 미리 적는다

- **가중 없는 BCE 기준판이다.** 희소 부위가 죽을 가능성이 높고, 그건 실패가 아니라
  15b(`pos_weight`)의 출발점이다. 두 개를 같이 바꾸면 원인을 못 가린다.
- **기준선 B(부위별 개별 이진 분류기)는 뺐다.** 부위 수 × 5겹 × 2구성이라 비용이 배가된다.
  P-1이 통과하면 **15c**로 큐잉해 "다중라벨 공유가 손해가 아닌가"를 따로 검정한다.
- **유도 2구성만 돌린다.** `{II}`(단일유도 웨어러블)·`{II,V1}`(텔레메트리)는 확장분이다.
- **PTB-XL 부위 라벨은 자동판독기 유래**일 수 있다 — 유도 규칙으로 붙었다면 P-3이
  쉽게 통과한다. 통과해도 그 가능성을 함께 보고한다.
- **여전히 급성이 아니다.** 부위는 배우지만 급성/만성은 이 데이터에 없다(실험14 한계와 동일).
